# <center>  **<span style="font-size:80px;">Exploración de los Datos</span>** </center>

In [ ]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import DatasetKeys
from src.config import Paths
Paths.init_project()

In [ ]:
from src.water2fraud.features.preprocessor import WaterPreprocessor
preprocessor = WaterPreprocessor()

# AMAEM

In [ ]:
df_amaem = pd.read_csv(Paths.PROC_CSV_AMAEM)

# Convertimos a periodo (YYYY-MM) directamente sobre la columna original
df_amaem[DatasetKeys.FECHA] = pd.to_datetime(df_amaem[DatasetKeys.FECHA]).dt.to_period('M')

# Instituto Nacional de Estadística (INE)

## Municipios

In [ ]:
from src.water2fraud.features.ine_tourism_processor import INETourismProcessor

### Matching Barrios - Municipios

Nuestro datos *INE* catalogan los datos a través de **municipios**. En cambio, nuestros datos *AMAEM* utilizan los **barrios**. Por tanto, deberemos asignar a cada barrio su correspondiente municipio. En caso de que no se encuentre el municipio exacto utilizaremos una ponderación de varios municipios.

Dado que tenemos el **Total de Viviendas Turísticas (municipio)**, podemos obtener el **Total de Viviendas Turísticas (barrio)** utilizando ponderaciones del estilo: 

**41-PLAYA DE SAN JUAN**
- 10% Alicante
- 10% Campello
- 5% San Juan Pueblo


In [ ]:
df_barrio = INETourismProcessor._map_mun2barrios()

### Expansión de datos con frecuencia mensual (Interpolación)

Dado que el INE solo recopila dos datos cada año, utilizaremos una interpolación lineal para conseguir rellenar estos valores faltantes de **Viviendas Turísticas**.

In [ ]:
df_amaem_dom, df_merge = INETourismProcessor._merge_domesticos_ine(df_amaem, df_barrio)
df_interpolated        = INETourismProcessor._interpolacion_mensual(df_merge)

### Porcentaje Viviendas Turísticas (INE + AMAEM)

Ahora que tenemos cada barrio mapeado con sus correspondientes **número de viviendas**, basado en las ponderaciones de los municipios asignados. Calcularemos el **porcentaje de viviendas turísticas** utilizando el **número de contratos domésticos** conocido. 

In [ ]:
df_final_mun = INETourismProcessor._porcentaje_vt(df_amaem_dom, df_interpolated)

### Resultados

In [ ]:
df_final_mun.isnull().sum()

In [ ]:
display(df_final_mun.head())

In [ ]:
df_maximos = df_final_mun.groupby(DatasetKeys.BARRIO)[DatasetKeys.PCT_VT_BARRIO].max().reset_index()
display(df_maximos)

## Provincia

In [ ]:
df_final_prov = INETourismProcessor._process_provincia()

In [ ]:
display(df_final_prov.head(10))